# AI651 PA1 Task 2

In [1]:
%%bash
set -e
cd /kaggle/working
BASE=$(dirname $(find /kaggle/input -name run.py | head -1))
echo "BASE = $BASE"
cp "$BASE/run.py" "$BASE/autoformer.py" .
rm -rf Data finC finA finCA && cp -r "$BASE/Data" .
grep -c "cov_head" run.py autoformer.py

BASE = /kaggle/input/datasets/amnafiaz555/pa1task2/pa1task2
run.py:2
autoformer.py:3


In [2]:
%%bash
set -e
cd /kaggle/working
python run.py --phase ablate --ext 1 --seeds 0 1 2 3 4 --stride 6 --epochs 8 --target raw --rich --hour --cov_head --out finC
python run.py --phase summarize --out finC
python run.py --phase final --final_seeds 0 1 2 3 4 --stride 6 --epochs 8 --target raw --rich --hour --cov_head --floor 5 --out finC
python run.py --phase analog --k 100 --out finA
python run.py --phase blend --dirs finC finA --out finCA

{'config': 'with external', 'seed': 0, 'params': 7778, 'epochs': 8, 'seconds': 20.5, 'RMSE': 61.873653411865234, 'MAE': 46.35912322998047, 'sMAPE': 66.9467544555664, 'block_rmse': [99.4815673828125, 81.8814468383789, 59.66584396362305, 57.82820129394531, 37.71213150024414, 58.399803161621094, 35.4321403503418, 32.14563751220703], 'rmse_by_epoch': [80.97, 64.23, 61.59, 61.48, 58.85, 61.82, 61.99, 61.87]}
{'config': 'with external', 'seed': 1, 'params': 7778, 'epochs': 8, 'seconds': 14.8, 'RMSE': 55.2653923034668, 'MAE': 41.039146423339844, 'sMAPE': 63.955623626708984, 'block_rmse': [90.5899658203125, 55.62238311767578, 53.57526397705078, 52.81379318237305, 41.31790542602539, 59.34968185424805, 35.95473861694336, 30.851472854614258], 'rmse_by_epoch': [77.09, 65.48, 63.38, 59.93, 59.37, 54.73, 53.84, 55.27]}
{'config': 'with external', 'seed': 2, 'params': 7778, 'epochs': 8, 'seconds': 14.7, 'RMSE': 60.511295318603516, 'MAE': 44.629737854003906, 'sMAPE': 67.37335968017578, 'block_rmse': [

In [3]:
import json, numpy as np, pandas as pd
runs = pd.read_csv("finC/ablation.csv"); runs = runs[runs.seed >= 0]
vt = np.load("finC/val_true.npy")
ens = np.mean([np.load(f"finC/val_pred_ext1_seed{s}.npy") for s in range(5)], 0)
rm = lambda p: float(np.sqrt(((p - vt) ** 2).mean())); ma = lambda p: float(np.abs(p - vt).mean())
sub = json.load(open("finCA/submission.json")); fc = np.array(sub["predictions"].split(","), float)
summary = {
  "single_seed_RMSE_mean": round(runs.RMSE.mean(), 1), "single_seed_RMSE_std": round(runs.RMSE.std(), 1),
  "single_seed_MAE_mean": round(runs.MAE.mean(), 1), "single_seed_sMAPE_mean": round(runs.sMAPE.mean(), 1),
  "ensemble_RMSE": round(rm(ens), 1), "ensemble_MAE": round(ma(ens), 1),
  "blend_config": sub["config"], "blend_val_RMSE": sub["validation_RMSE"],
  "P": sub["P"], "E": sub["E"],
  "forecast_mean": round(fc.mean(), 1), "forecast_min": round(fc.min(), 1), "forecast_max": round(fc.max(), 1),
  "blend_table": pd.read_csv("finCA/blend_weights.csv").round(2).to_dict("records"),
}
json.dump(summary, open("finCA/report_numbers.json", "w"), indent=1)
print(json.dumps({k: v for k, v in summary.items() if k != "blend_table"}, indent=1))
print(pd.read_csv("finCA/blend_weights.csv").round(2).to_string(index=False))
print("\nPREDICTIONS (paste on leaderboard with P and E above):\n" + sub["predictions"])

{
 "single_seed_RMSE_mean": 60.8,
 "single_seed_RMSE_std": 3.3,
 "single_seed_MAE_mean": 46.0,
 "single_seed_sMAPE_mean": 68.0,
 "ensemble_RMSE": 57.5,
 "ensemble_MAE": 43.1,
 "blend_config": "blend 0.8*finC + 0.2*finA",
 "blend_val_RMSE": 56.04,
 "P": 38890,
 "E": 40,
 "forecast_mean": 112.0,
 "forecast_min": 7.1,
 "forecast_max": 236.7
}
 weight_first  RMSE   MAE  sMAPE
          0.0 72.75 51.34  61.57
          0.1 69.12 49.25  61.41
          0.2 65.83 47.34  61.27
          0.3 62.93 45.67  61.21
          0.4 60.46 44.30  61.31
          0.5 58.49 43.34  61.65
          0.6 57.08 42.70  62.14
          0.7 56.25 42.43  62.75
          0.8 56.04 42.39  63.07
          0.9 56.45 42.58  63.33
          1.0 57.47 43.14  63.60

PREDICTIONS (paste on leaderboard with P and E above):
20.4196,17.7502,14.7266,14.6648,11.5508,10.8530,31.2000,21.5720,17.8180,11.0580,9.7260,14.7960,12.6900,10.8760,21.1195,21.2158,36.7147,47.3157,62.3846,79.9818,85.4057,94.6062,95.7051,91.6541,89.1636,97.9324

In [4]:
%%bash
cd /kaggle/working
rm -f task2_results.zip && zip -qr task2_results.zip finC finA finCA run.py autoformer.py && ls -lh task2_results.zip

-rw-r--r-- 1 root root 50K Oct  5 07:24 task2_results.zip
